# Imports and Configs

In [ ]:
# Install klib and flash libraries
!pip install klib git+https://github.com/flash-lib/flash.git -q

In [ ]:
# Standard Libraries
import toml

# Data Manipulation
import numpy as np
import pandas as pd

# Data Preprocessing
import klib
import flash as fz

# Scikit-learn
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, LabelEncoder, RobustScaler, OneHotEncoder
)

# Machine Learning Models
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier,
    VotingClassifier
    )

# Saving
import joblib

In [ ]:
# Change directory to current working directory
%cd /content/drive/MyDrive/Projects/loan-sanction-prediction

In [ ]:
# Load the configurations
with open("config/config_v2.toml", "r") as file:
    config_data = toml.load(file)

num_cols, cat_cols = config_data['num']['cols'], config_data['cat']['cols']

# Model Building

In [ ]:
# Load the dataset
df = pd.read_csv('data/interim/feature_engineered_train_data_v1.csv')

# Split the data into features and target
X = df.drop('loan_status', axis=1)
y = df['loan_status']

In [ ]:
# Label encode target
le = LabelEncoder()
y = le.fit_transform(y)

In [ ]:
# Tranformer for preprocessing data
transformer = make_column_transformer(
    (StandardScaler(), num_cols),
    (OneHotEncoder(drop='first', sparse_output=False), cat_cols),
    remainder='passthrough'
)

In [ ]:
X_transformed = transformer.fit_transform(X)

# Test
X_transformed.shape

## Handling imbalanced dataset

In [ ]:
# Oversampling the dataset using SMOTE
smote = SMOTE(random_state=42)
X_transformed, y_resampled = smote.fit_resample(X_transformed, y)

# Test
print(X_transformed.shape, y_resampled.shape)

In [ ]:
# Test
unique_values, counts = np.unique(y_resampled, return_counts=True)

# Print the counts of each class
for value, count in zip(unique_values, counts):
    print(f"Class {value}: {count}")

## Model selection (Before hyperparameter tuning)

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(),
    'Random Forest': RandomForestClassifier(),
    'Gradient Boosting': GradientBoostingClassifier(),
    'Support Vector Machine': SVC(),
    'KNN': KNeighborsClassifier(),
    'Decision Trees': DecisionTreeClassifier(),
    'Xgboost': XGBClassifier(),
    'Extra Trees': ExtraTreesClassifier()
}

In [ ]:
# Define metric functions
metrics = {
    'accuracy': accuracy_score,
    'precision': precision_score,
    'recall': recall_score,
    'f1': f1_score
}

In [ ]:
def eval_models_across_metrics(models, metrics, X, y, cv=5):
    models_across_metrics = {metric: {} for metric in metrics}
    for metric in metrics:
        for model_name, model in models.items():
            cv_scores = cross_val_score(model, X, y, cv=cv, scoring=metric)
            cv_scores_mean = cv_scores.mean()
            models_across_metrics[metric][model_name] = round(cv_scores_mean, 3)
    return pd.DataFrame(models_across_metrics)

In [ ]:
models_across_metrics = eval_models_across_metrics(models, metrics.keys(), X_transformed, y_resampled)

In [ ]:
models_across_metrics

Conclusions:

- After evaluating the metrics, I have decided to focus on the top 3 models (in terms of accuracy_score): Random Forest Classifier, Extra Trees Classifier, Xgboost Classifier.

## Hyperparameter tuning

In [ ]:
# Define top models for further hyperparameter tuning
models = {
    'Random Forest': RandomForestClassifier(),
    'Xgboost': XGBClassifier(),
    'Extra Trees': ExtraTreesClassifier()
}

In [ ]:
# Define parameter grids
param_grids = {
    'Random Forest': {
        'n_estimators': [50, 200],
        'max_depth': [None, 30],
        'min_samples_split': [2, 10],
        'min_samples_leaf': [1, 4]
    },
    'Xgboost': {
        'n_estimators': [50, 200],
        'max_depth': [3, 10],
        'learning_rate': [0.01, 0.2],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0],
        'gamma': [0, 0.2]
    },
    'Extra Trees': {
        'n_estimators': [50, 200],
        'max_depth': [None, 30],
        'min_samples_split': [2, 10],
        'min_samples_leaf': [1, 4],
        'bootstrap': [True, False]
    }
}

In [ ]:
def perform_grid_search(models, param_grids, X, y):
    best_params = {}
    for model_name, model in models.items():
        print(f"Processing {model_name}...")
        param_grid = param_grids[model_name]
        grid_search = GridSearchCV(
            estimator=model, param_grid=param_grid, scoring='accuracy', cv=5, n_jobs=-1,
            verbose=1
            )
        grid_search.fit(X, y)
        best_params[model_name] = {
            'Best Parameters': grid_search.best_params_,
            'Average accuracy score on the best parameters': round(grid_search.best_score_, 3)
        }
    return best_params

In [ ]:
# Finding best hyperparameters on top models using GridSearchCV
best_params = perform_grid_search(models, param_grids, X_transformed, y_resampled)

In [ ]:
pd.DataFrame(best_params)

In [ ]:
# best_params

# {'Random Forest': {'Best Parameters': {'max_depth': None,
#    'min_samples_leaf': 1,
#    'min_samples_split': 2,
#    'n_estimators': 200},
#   'Average accuracy score on the best parameters': 0.838},
#  'Xgboost': {'Best Parameters': {'colsample_bytree': 0.8,
#    'gamma': 0.2,
#    'learning_rate': 0.2,
#    'max_depth': 10,
#    'n_estimators': 50,
#    'subsample': 1.0},
#   'Average accuracy score on the best parameters': 0.826},
#  'Extra Trees': {'Best Parameters': {'bootstrap': True,
#    'max_depth': 30,
#    'min_samples_leaf': 1,
#    'min_samples_split': 2,
#    'n_estimators': 200},
#   'Average accuracy score on the best parameters': 0.84}}

In [ ]:
# Define top models with best hyperparameters
models = {
    'Random Forest': RandomForestClassifier(**best_params['Random Forest']['Best Parameters']),
    'Xgboost': XGBClassifier(**best_params['Xgboost']['Best Parameters']),
    'Extra Trees': ExtraTreesClassifier(**best_params['Extra Trees']['Best Parameters'])
}

In [ ]:
# Comparing top models across metrics after hyperparameter tuning
models_across_metrics = eval_models_across_metrics(models, metrics.keys(), X_transformed, y_resampled)

In [ ]:
models_across_metrics


## Model training


In [ ]:
estimators = []
for model_name, model in models.items():
    estimators.append((model_name, model))

In [ ]:
estimators

In [ ]:
def eval_voting_clf(estimators, X, y, cv = 5):
    # Create a voting classifier (hard voting)
    voting_clf_hard = VotingClassifier(estimators=estimators, voting='hard')

    # Create a voting classifier (soft voting)
    voting_clf_soft = VotingClassifier(estimators=estimators, voting='soft')

    # Apply cross-validation
    cv_scores_h = cross_val_score(voting_clf_hard, X, y, cv=cv, scoring='accuracy')
    cv_scores_s = cross_val_score(voting_clf_soft, X, y, cv=cv, scoring='accuracy')

    accuracy_results = {}

    accuracy_results['Hard Margin'] = round(cv_scores_h.mean(), 3)
    accuracy_results['Soft Margin'] = round(cv_scores_s.mean(), 3)

    return accuracy_results

In [ ]:
# Accuracy on hard and soft margin voting classifiers
accuracy = eval_voting_clf(estimators, X_transformed, y_resampled)
accuracy

In [ ]:
# Fit the best model
voting_clf = VotingClassifier(estimators, voting='hard')
voting_clf.fit(X_transformed, y_resampled)

In [ ]:
# Create a pipeline
pipeline = make_pipeline(
    StandardScaler(),     # Step 1: Standardize the data
    PCA(n_components=2),  # Step 2: Apply PCA
    LogisticRegression()  # Step 3: Train a logistic regression model
)


In [ ]:
# Pipeline
pipe = make_pipeline(
    (transformer),
    ('model', voting_clf)
)

pipe.fit(X, y)

## Saving

In [ ]:
# Save the Machine Learning model
joblib.dump(voting_clf, 'model.joblib')